# Imports

In [1]:
import pdfplumber
import re
import os
import sqlite3
import json
from collections import Counter

# Exploring the data — scan all PDFs, find all table types

In [12]:
PDF_FOLDER = r"C:\Users\asule\Desktop\Task_DS\PDF_version_1000"
all_pdfs = sorted([f for f in os.listdir(PDF_FOLDER) if f.endswith('.pdf')])

print(f"Scanning {len(all_pdfs)} PDFs for table types...")

all_types = Counter()
failed = []

for i, filename in enumerate(all_pdfs):
    pdf_path = os.path.join(PDF_FOLDER, filename)
    try:
        with pdfplumber.open(pdf_path) as pdf:
            all_tables = pdf.pages[0].extract_tables()
            if len(pdf.pages) > 1:
                all_tables += pdf.pages[1].extract_tables()

        for table in all_tables:
            ttype = get_table_type(table)
            all_types[ttype] += 1

        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{len(all_pdfs)}...")

    except Exception as e:
        failed.append((filename, str(e)))

print(f"\n=== TABLE TYPES FOUND ACROSS ALL {len(all_pdfs)} PDFs ===")
for ttype, count in all_types.most_common():
    print(f"  {count:5d} tables — {ttype}")

print(f"\nTotal unique table types: {len(all_types)}")
print(f"Failed PDFs: {len(failed)}")

Scanning 1000 PDFs for table types...
  Progress: 100/1000...
  Progress: 200/1000...
  Progress: 300/1000...
  Progress: 400/1000...
  Progress: 500/1000...
  Progress: 600/1000...
  Progress: 700/1000...
  Progress: 800/1000...
  Progress: 900/1000...
  Progress: 1000/1000...

=== TABLE TYPES FOUND ACROSS ALL 1000 PDFs ===
   1026 tables — operations
   1020 tables — drilling_fluid
   1000 tables — title
   1000 tables — header_col1
   1000 tables — header_col3
    991 tables — header_col2
    494 tables — unknown
    475 tables — pore_pressure
    186 tables — survey
    178 tables — lithology
    148 tables — equipment_failure
    127 tables — gas_reading
     56 tables — stratigraphic

Total unique table types: 13
Failed PDFs: 0


In [2]:
# NOTE: 494 tables were classified as 'unknown' across all 1000 PDFs.
# After manual inspection of sample PDFs, these were identified as:
# - Continuation rows of Drilling Fluid table split across pages (e.g. 'Pm filtrate', 'Filtrate Lthp')
# - Stray/empty cells artifacts from PDF rendering
# - Small fragments from table splitting at page breaks
# These do not represent meaningful new sections and are safely ignored.
# All 11 meaningful section types are fully extracted:
# operations, drilling_fluid, title, header_col1, header_col2, header_col3,
# pore_pressure, survey, lithology, equipment_failure, gas_reading, stratigraphic

# Helper Functions

In [3]:
def fix_doubled(text):
    """Fix font artifact where each character is doubled.
    e.g. 'WWeellllbboorree' -> 'Wellbore'
    Skips digits to avoid breaking values like '1997', '00:00'
    """
    if not text:
        return text
    result = []
    i = 0
    while i < len(text):
        result.append(text[i])
        if i + 1 < len(text) and text[i] == text[i+1] and not text[i].isdigit():
            i += 2
        else:
            i += 1
    return ''.join(result)

def clean_val(val):
    """
    Normalize cell values:
    - Remove newlines (fixes split words like 'con\nditioning')
    - Convert -999.99 sentinel and empty strings to None
    """
    if val is None:
        return None
    v = str(val).replace('\n', '')
    v = re.sub(r'\s+', ' ', v).strip()
    if v in ('', '-999.99', '-999.9', '-999'):
        return None
    return v

# Table Type Detector

In [4]:
def get_table_type(table):

    """
    Identify table section type by content, not by index.
    This is robust to variations in table count across PDFs.

    WHY content-based: structural scan showed table counts vary
    between 4-8 per PDF, so hardcoded indices would break.
    """
    # Build both raw and fixed versions for different checks
    # Raw: needed for labels that don't have doubled chars (e.g. 'Status:', 'Sample Time')
    # Fixed: needed for headers with doubled chars (e.g. 'SSttaarrtt' -> 'Start')

    all_text_raw = ' '.join(
        str(cell) for row in table for cell in row if cell
    )
    all_text_fixed = ' '.join(
        fix_doubled(str(cell)) for row in table for cell in row if cell
    )

    first_raw = ''
    for row in table:
        for cell in row:
            if cell and str(cell).strip():
                first_raw = str(cell).strip().rstrip(':').strip()
                break
        if first_raw:
            break


    first_fixed = fix_doubled(first_raw)
    first_row_fixed = ''
    for row in table:
        if any(cell for cell in row if cell and str(cell).strip()):
            first_row_fixed = fix_doubled(' '.join(
                str(c) for c in row if c and str(c).strip()
            ))
            break

    if 'Summary report' in first_raw:                                   return 'title'
    if 'Status' in first_raw:                                           return 'header_col1'
    if 'Dist Drilled' in first_raw:                                     return 'header_col2'
    if 'Depth at Kick' in first_raw:                                    return 'header_col3'
    if 'Sample Time' in first_raw:                                      return 'drilling_fluid'
    if 'Mf ()' in first_raw or ('Plastic visc' in all_text_fixed
                              and 'Sample Time' not in all_text_raw):   return 'drilling_fluid'


    if 'Pore Pressure' in first_fixed:                                  return 'pore_pressure'
    if 'Depth to Top of Formation' in first_fixed:                      return 'stratigraphic'

    if 'Start' in first_row_fixed and 'time' in first_row_fixed.lower():
        if ('Downtime' in all_text_fixed or 'Equip' in all_text_fixed) \
                and 'Activity' not in all_text_fixed:                   return 'equipment_failure'
        else:                                                           return 'operations'


    if 'Depth mMD' in all_text_fixed and 'Inclination' in all_text_fixed: return 'survey'
    if 'Start Depth' in first_fixed:                                    return 'lithology'
    if 'Class' in all_text_fixed and 'Highest Gas' in all_text_fixed:  return 'gas_reading'


    if 'Time' in first_fixed:
        if 'Equ Mud Weight' in all_text_fixed:                          return 'pore_pressure'
        if 'Class' in all_text_fixed:                                   return 'gas_reading'

    return 'unknown'

# Section parsers

In [5]:
def parse_header_tables(t1, t2, t3, t4, words):

    """
    Header is split across 4 tables + a free-text line.
    Wellbore and Period are on a free-text line (not in any table),
    extracted using word x-position: Wellbore x<300, Period x>=300.
    Tables t2, t3, t4 are the 3 metadata columns.
    """

    header = {}
    lines = {}
    for w in words:
        y = round(w['top'])
        lines.setdefault(y, []).append(w)

    for y, ws in sorted(lines.items()):
        ws_sorted = sorted(ws, key=lambda x: x['x0'])
        left  = [fix_doubled(w['text']) for w in ws_sorted if w['x0'] < 300]
        right = [fix_doubled(w['text']) for w in ws_sorted if w['x0'] >= 300]
        left_text  = ' '.join(left)
        right_text = ' '.join(right)
        m = re.match(r'Wellbore:+\s*(.+)', left_text)
        if m:
            header['Wellbore'] = m.group(1).strip()
        m = re.match(r'Period:+\s*(.+)', right_text)
        if m:
            header['Period'] = m.group(1).strip()
        if 'Wellbore' in header and 'Period' in header:
            break

    if 'Period' not in header and t1 and len(t1) >= 2:
        cell = t1[1][1] if t1[1][1] else ''
        fixed = fix_doubled(cell)
        m = re.match(r'Period:+\s*(.+)', fixed)
        if m:
            header['Period'] = m.group(1).strip()

    for t in [t2, t3, t4]:
        if not t:
            continue
        for row in t:
            if not row or not row[0]:
                continue
            key = row[0].strip().rstrip(':').strip()
            val = clean_val(row[1]) if len(row) > 1 else None
            if key and len(key) >= 2:
                header[key] = val
    return header


def parse_operations(table):

    # Skip empty leading rows — some PDFs have blank first row before header

    header_idx = None
    for i, row in enumerate(table):
        if any(cell for cell in row if cell and str(cell).strip()):
            header_idx = i
            break
    if header_idx is None:
        return []
    headers = [fix_doubled(c).replace('\n', ' ').strip() if c else ''
               for c in table[header_idx]]
    ops = []
    for row in table[header_idx + 1:]:
        if not any(row):
            continue
        record = {headers[i] if i < len(headers) else f'col_{i}': clean_val(cell)
                  for i, cell in enumerate(row)}
        ops.append(record)
    return ops


def parse_equipment_failure(table):
    header_idx = None
    for i, row in enumerate(table):
        if any(cell for cell in row if cell and str(cell).strip()):
            header_idx = i
            break
    if header_idx is None:
        return []
    headers = [fix_doubled(c).replace('\n', ' ').strip() if c else ''
               for c in table[header_idx]]
    KEEP = {
        'Start time', 'Depth mMD', 'Depth mTVD',
        'Sub Equip - Syst Class', 'Operation Downtime (min)', 'Remark'
    }
    rows = []
    for row in table[header_idx + 1:]:
        if not any(row):
            continue
        record = {}
        for i, cell in enumerate(row):
            col = headers[i] if i < len(headers) else f'col_{i}'
            if col in KEEP:
                record[col] = clean_val(cell)
        rows.append(record)
    return rows


def parse_drilling_fluid(tables_list):

    """
    Drilling Fluid can be split across 2 pages — we collect all fluid
    tables and merge them. Some PDFs have 2 samples, some have 4.
    We store all sample values as a list regardless of count.
    """

    KEEP = {
        'Sample Time', 'Sample Depth mMD', 'Fluid Type',
        'Fluid Density (g/cm3)', 'Plastic visc. (mPa.s)', 'Yield point (Pa)'
    }
    SUBHEADERS = {'Solids', 'Viscometer tests', 'Filtration tests'}

    fluid = {}
    for table in tables_list:
        for row in table:
            if not row or not row[0]:
                continue
            key = row[0].strip()
            if key in SUBHEADERS or key not in KEEP:
                continue
            # Handle any number of samples (2, 4, or more)
            # Store all values as a list
            vals = [clean_val(row[j]) for j in range(1, len(row)) if j < len(row)]
            # Remove trailing Nones
            while vals and vals[-1] is None:
                vals.pop()
            fluid[key] = vals if vals else [None]

    return fluid


def parse_generic_table(table):

    """
    Generic parser for: Pore Pressure, Survey Station,
    Lithology, Gas Reading, Stratigraphic.
    Finds header row dynamically (skips empty leading rows).
    Fixes doubled chars and doubled digits in column names.
    """

    header_idx = None
    for i, row in enumerate(table):
        if any(cell for cell in row if cell and str(cell).strip()):
            header_idx = i
            break
    if header_idx is None:
        return []

    def clean_header(c):
        if not c:
            return ''
        fixed = fix_doubled(c)
        fixed = fixed.replace('\n', ' ').strip()
        # Fix doubled digits in column names e.g. C11->C1, g/cm33->g/cm3, IC44->IC4
        fixed = re.sub(r'(\d)\1', r'\1', fixed)
        return fixed

    headers = [clean_header(c) for c in table[header_idx]]
    rows = []
    for row in table[header_idx + 1:]:
        if not any(row):
            continue
        record = {headers[i] if i < len(headers) else f'col_{i}': clean_val(cell)
                  for i, cell in enumerate(row)}
        rows.append(record)
    return rows


def parse_summaries(pdf_path):

    """Extract free-text summary sections using regex on full page text."""

    all_text = ''
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            all_text += (page.extract_text() or '') + '\n'
    summary_24h = summary_planned = None
    m = re.search(
        r'Summary of activities \(24 Hours?\)(.*?)Summary of planned activities',
        all_text, re.DOTALL | re.IGNORECASE)
    if m:
        summary_24h = m.group(1).strip()
    m = re.search(
        r'Summary of planned activities \(24 Hours?\)(.*?)(?:Operations|$)',
        all_text, re.DOTALL | re.IGNORECASE)
    if m:
        summary_planned = m.group(1).strip()
    return summary_24h, summary_planned

#  Main parse_pdf function

In [6]:
def parse_pdf(pdf_path):

    """
    Main entry point. Parses all sections from a drilling report PDF.
    Tables are routed by content type, not by index — robust to
    structural variations across 1000 PDFs.
    """

    with pdfplumber.open(pdf_path) as pdf:
        page1 = pdf.pages[0]
        all_tables = page1.extract_tables()
        words = page1.extract_words()
        if len(pdf.pages) > 1:
            all_tables += pdf.pages[1].extract_tables()

    t_title = t_h1 = t_h2 = t_h3 = None
    operations = []
    drilling_fluid_tables = []
    pore_pressure = []
    survey_station = []
    lithology = []
    gas_reading = []
    equipment_failure = []
    stratigraphic = []

    # Route each table to its parser by detected type
    for table in all_tables:
        ttype = get_table_type(table)
        if ttype == 'title':               t_title = table
        elif ttype == 'header_col1':       t_h1 = table
        elif ttype == 'header_col2':       t_h2 = table
        elif ttype == 'header_col3':       t_h3 = table
        elif ttype == 'operations':        operations = parse_operations(table)
        elif ttype == 'drilling_fluid':    drilling_fluid_tables.append(table)
        elif ttype == 'equipment_failure': equipment_failure = parse_equipment_failure(table)
        elif ttype == 'pore_pressure':     pore_pressure = parse_generic_table(table)
        elif ttype == 'survey':            survey_station = parse_generic_table(table)
        elif ttype == 'lithology':         lithology = parse_generic_table(table)
        elif ttype == 'gas_reading':       gas_reading = parse_generic_table(table)
        elif ttype == 'stratigraphic':     stratigraphic = parse_generic_table(table)  # NEW

    header = parse_header_tables(t_title, t_h1, t_h2, t_h3, words)
    summary_24h, summary_planned = parse_summaries(pdf_path)
    drilling_fluid = parse_drilling_fluid(drilling_fluid_tables)

    return {
        'source_file': os.path.basename(pdf_path),
        'header': header,
        'summary_24h': summary_24h,
        'summary_planned': summary_planned,
        'operations': operations,
        'equipment_failure': equipment_failure,
        'drilling_fluid': drilling_fluid,
        'pore_pressure': pore_pressure,
        'survey_station': survey_station,
        'lithology': lithology,
        'gas_reading': gas_reading,
        'stratigraphic': stratigraphic,
    }

# Test with one pdf

In [7]:
pdf_name = r"C:\Users\asule\Desktop\Task_DS\PDF_version_1000\15_9_F_15_A_2008_12_15.pdf"

print(f"\n{'='*60}")
print(f"FILE: {os.path.basename(pdf_name)}")
result = parse_pdf(pdf_name)

print("HEADER:")
for k, v in result['header'].items():
  print(f"  {k:<45} = {repr(v)}")

print(f"\nSUMMARY 24H:\n  {result['summary_24h']}")
print(f"\nSUMMARY PLANNED:\n  {result['summary_planned']}")

print(f"\nOPERATIONS ({len(result['operations'])} rows):")
for op in result['operations']:
  print(f"  {op}")

print(f"\nEQUIPMENT FAILURE ({len(result['equipment_failure'])} rows):")
for row in result['equipment_failure']:
  print(f"  {row}")

print(f"\nDRILLING FLUID:")
for k, v in result['drilling_fluid'].items():
    print(f"  {k:<35} = {v}")

print(f"\nPORE PRESSURE ({len(result['pore_pressure'])} rows):")
for row in result['pore_pressure']:
  print(f"  {row}")

print(f"\nSURVEY STATION ({len(result['survey_station'])} rows):")
for row in result['survey_station']:
  print(f"  {row}")

print(f"\nLITHOLOGY ({len(result['lithology'])} rows):")
for row in result['lithology']:
  print(f"  {row}")

print(f"\nGAS READING ({len(result['gas_reading'])} rows):")
for row in result['gas_reading']:
  print(f"  {row}")

print(f"\nSTRATIGRAPHIC ({len(result['stratigraphic'])} rows):")
for row in result['stratigraphic']:
  print(f"  {row}")


FILE: 15_9_F_15_A_2008_12_15.pdf
HEADER:
  Wellbore                                      = '15/9-F-15 A'
  Period                                        = '2008-12-14 00:00 - 2008-12-15 00:00'
  Status                                        = 'normal'
  Report creation time                          = '2018-05-03 13:52'
  Report number                                 = '4'
  Days Ahead/Behind (+/-)                       = None
  Operator                                      = 'StatoilHydro'
  Rig Name                                      = 'MÆRSK INSPIRER'
  Drilling contractor                           = 'Mærsk Contractors'
  Spud Date                                     = '2008-12-11 15:00'
  Wellbore type                                 = None
  Elevation RKB-MSL (m)                         = '54.9'
  Water depth MSL (m)                           = '91'
  Tight well                                    = 'Y'
  HPHT                                          = 'Y'
  Temperature ()       

# Creating database

In [9]:
DB_PATH = r"C:\Users\asule\Desktop\Task_DS\drilling_reports.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS stratigraphic;
DROP TABLE IF EXISTS gas_reading;
DROP TABLE IF EXISTS lithology;
DROP TABLE IF EXISTS survey_station;
DROP TABLE IF EXISTS pore_pressure;
DROP TABLE IF EXISTS drilling_fluid;
DROP TABLE IF EXISTS equipment_failure;
DROP TABLE IF EXISTS operations;
DROP TABLE IF EXISTS reports;

CREATE TABLE reports (
    id                          INTEGER PRIMARY KEY AUTOINCREMENT,
    source_file                 TEXT,
    wellbore                    TEXT,
    period                      TEXT,
    status                      TEXT,
    report_number               TEXT,
    report_creation_time        TEXT,
    operator                    TEXT,
    rig_name                    TEXT,
    drilling_contractor         TEXT,
    spud_date                   TEXT,
    water_depth_msl             TEXT,
    elevation_rkb_msl           TEXT,
    tight_well                  TEXT,
    hpht                        TEXT,
    hole_dia                    TEXT,
    pressure_test_type          TEXT,
    formation_strength          TEXT,
    depth_mmd                   TEXT,
    depth_mtvd                  TEXT,
    depth_kick_off_mmd          TEXT,
    depth_kick_off_mtvd         TEXT,
    depth_last_casing_mmd       TEXT,
    depth_last_casing_mtvd      TEXT,
    depth_formation_strength_mmd  TEXT,
    depth_formation_strength_mtvd TEXT,
    plug_back_depth_mmd         TEXT,
    summary_24h                 TEXT,
    summary_planned             TEXT
);

CREATE TABLE operations (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id           INTEGER,
    source_file         TEXT,
    start_time          TEXT,
    end_time            TEXT,
    end_depth_mmd       TEXT,
    main_sub_activity   TEXT,
    state               TEXT,
    remark              TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE equipment_failure (
    id                      INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id               INTEGER,
    source_file             TEXT,
    start_time              TEXT,
    depth_mmd               TEXT,
    depth_mtvd              TEXT,
    sub_equip_syst_class    TEXT,
    operation_downtime_min  TEXT,
    remark                  TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE drilling_fluid (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id           INTEGER,
    source_file         TEXT,
    sample_time         TEXT,
    sample_depth_mmd    TEXT,
    fluid_type          TEXT,
    fluid_density       TEXT,
    plastic_visc        TEXT,
    yield_point         TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE pore_pressure (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id           INTEGER,
    source_file         TEXT,
    time                TEXT,
    depth_mmd           TEXT,
    depth_tvd           TEXT,
    equ_mud_weight      TEXT,
    reading             TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE survey_station (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id     INTEGER,
    source_file   TEXT,
    depth_mmd     TEXT,
    depth_mtvd    TEXT,
    inclination   TEXT,
    azimuth       TEXT,
    comment       TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE lithology (
    id                    INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id             INTEGER,
    source_file           TEXT,
    start_depth_mmd       TEXT,
    end_depth_mmd         TEXT,
    start_depth_tvd       TEXT,
    end_depth_tvd         TEXT,
    shows_description     TEXT,
    lithology_description TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE gas_reading (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id           INTEGER,
    source_file         TEXT,
    time                TEXT,
    class_              TEXT,
    depth_top_mmd       TEXT,
    depth_bottom_mmd    TEXT,
    depth_top_tvd       TEXT,
    depth_bottom_tvd    TEXT,
    highest_gas         TEXT,
    lowest_gas          TEXT,
    c1                  TEXT,
    c2                  TEXT,
    c3                  TEXT,
    ic4                 TEXT,
    ic5                 TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);

CREATE TABLE stratigraphic (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id       INTEGER,
    source_file     TEXT,
    depth_top_mmd   TEXT,
    depth_top_mtvd  TEXT,
    description     TEXT,
    FOREIGN KEY (report_id) REFERENCES reports(id)
);
""")

conn.commit()
print("Database created with 9 tables!")

Database created with 9 tables!


# Inserting parsed sections

In [10]:
def insert_to_db(conn, result):
    """Insert all parsed sections of one PDF into the database."""
    cur = conn.cursor()
    h = result['header']

    # Insert main report metadata
    cur.execute("""
        INSERT INTO reports (
            source_file, wellbore, period, status, report_number,
            report_creation_time, operator, rig_name, drilling_contractor,
            spud_date, water_depth_msl, elevation_rkb_msl, tight_well, hpht,
            hole_dia, pressure_test_type, formation_strength,
            depth_mmd, depth_mtvd, depth_kick_off_mmd, depth_kick_off_mtvd,
            depth_last_casing_mmd, depth_last_casing_mtvd,
            depth_formation_strength_mmd, depth_formation_strength_mtvd,
            plug_back_depth_mmd, summary_24h, summary_planned
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        result['source_file'],
        h.get('Wellbore'),
        h.get('Period'),
        h.get('Status'),
        h.get('Report number'),
        h.get('Report creation time'),
        h.get('Operator'),
        h.get('Rig Name'),
        h.get('Drilling contractor'),
        h.get('Spud Date'),
        h.get('Water depth MSL (m)'),
        h.get('Elevation RKB-MSL (m)'),
        h.get('Tight well'),
        h.get('HPHT'),
        h.get('Hole Dia (in)'),
        h.get('Pressure Test Type'),
        h.get('Formation strength (g/cm3)'),
        h.get('Depth mMd'),
        h.get('Depth mTVD'),
        h.get('Depth at Kick Off mMD'),
        h.get('Depth at Kick Off mTVD'),
        h.get('Depth At Last Casing mMD'),
        h.get('Depth At Last Casing mTVD'),
        h.get('Depth at formation strength mMD'),
        h.get('Depth At Formation Strength mTVD'),
        h.get('Plug Back Depth mMD'),
        result['summary_24h'],
        result['summary_planned'],
    ))
    report_id = cur.lastrowid

    # Operations
    for op in result['operations']:
        cur.execute("""
            INSERT INTO operations (
                report_id, source_file, start_time, end_time,
                end_depth_mmd, main_sub_activity, state, remark
            ) VALUES (?,?,?,?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            op.get('Start time'), op.get('End time'),
            op.get('End Depth mMD'), op.get('Main - Sub Activity'),
            op.get('State'), op.get('Remark'),
        ))

    # Equipment failure
    for eq in result['equipment_failure']:
        cur.execute("""
            INSERT INTO equipment_failure (
                report_id, source_file, start_time, depth_mmd,
                depth_mtvd, sub_equip_syst_class, operation_downtime_min, remark
            ) VALUES (?,?,?,?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            eq.get('Start time'), eq.get('Depth mMD'),
            eq.get('Depth mTVD'), eq.get('Sub Equip - Syst Class'),
            eq.get('Operation Downtime (min)'), eq.get('Remark'),
        ))

    # Drilling fluid — one row per sample
    fluid = result['drilling_fluid']
    if fluid:
        times       = fluid.get('Sample Time',           [None])
        depths      = fluid.get('Sample Depth mMD',      [None])
        ftypes      = fluid.get('Fluid Type',             [None])
        densities   = fluid.get('Fluid Density (g/cm3)', [None])
        viscosities = fluid.get('Plastic visc. (mPa.s)', [None])
        yields      = fluid.get('Yield point (Pa)',       [None])
        n_samples   = max(len(times), len(depths), len(ftypes))
        for i in range(n_samples):
            cur.execute("""
                INSERT INTO drilling_fluid (
                    report_id, source_file, sample_time, sample_depth_mmd,
                    fluid_type, fluid_density, plastic_visc, yield_point
                ) VALUES (?,?,?,?,?,?,?,?)
            """, (
                report_id, result['source_file'],
                times[i]       if i < len(times)       else None,
                depths[i]      if i < len(depths)      else None,
                ftypes[i]      if i < len(ftypes)      else None,
                densities[i]   if i < len(densities)   else None,
                viscosities[i] if i < len(viscosities) else None,
                yields[i]      if i < len(yields)       else None,
            ))

    # Pore pressure
    for row in result['pore_pressure']:
        cur.execute("""
            INSERT INTO pore_pressure (
                report_id, source_file, time, depth_mmd,
                depth_tvd, equ_mud_weight, reading
            ) VALUES (?,?,?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            row.get('Time'), row.get('Depth mMD'),
            row.get('Depth TVD'), row.get('Equ Mud Weight (g/cm3)'),
            row.get('Reading'),
        ))

    # Survey station
    for row in result['survey_station']:
        cur.execute("""
            INSERT INTO survey_station (
                report_id, source_file, depth_mmd, depth_mtvd,
                inclination, azimuth, comment
            ) VALUES (?,?,?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            row.get('Depth mMD'), row.get('Depth mTVD'),
            row.get('Inclination (dega)'), row.get('Azimuth (dega)'),
            row.get('Comment'),
        ))

    # Lithology
    for row in result['lithology']:
        cur.execute("""
            INSERT INTO lithology (
                report_id, source_file, start_depth_mmd, end_depth_mmd,
                start_depth_tvd, end_depth_tvd, shows_description, lithology_description
            ) VALUES (?,?,?,?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            row.get('Start Depth mMD'), row.get('End Depth mMD'),
            row.get('Start Depth mTVD'), row.get('End Depth mTVD'),
            row.get('Shows Description'), row.get('Lithology Description'),
        ))

    # Gas reading
    for row in result['gas_reading']:
        cur.execute("""
            INSERT INTO gas_reading (
                report_id, source_file, time, class_,
                depth_top_mmd, depth_bottom_mmd,
                depth_top_tvd, depth_bottom_tvd,
                highest_gas, lowest_gas,
                c1, c2, c3, ic4, ic5
            ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            row.get('Time'), row.get('Class'),
            row.get('Depth to Top mMD'), row.get('Depth to Bottom MD'),
            row.get('Depth to Top mTVD'), row.get('Depth to Bottom TVD'),
            row.get('Highest Gas (%)'), row.get('Lowest Gas ()'),
            row.get('C1 (ppm)'), row.get('C2 (ppm)'),
            row.get('C3 (ppm)'), row.get('IC4 (ppm)'),
            row.get('IC5 (ppm)'),
        ))

    # Stratigraphic
    for row in result['stratigraphic']:
        cur.execute("""
            INSERT INTO stratigraphic (
                report_id, source_file, depth_top_mmd,
                depth_top_mtvd, description
            ) VALUES (?,?,?,?,?)
        """, (
            report_id, result['source_file'],
            row.get('Depth to Top of Formation mMD'),
            row.get('Depth to Top of Formation mTVD'),
            row.get('Description'),
        ))

    conn.commit()

In [11]:
PDF_FOLDER = r"C:\Users\asule\Desktop\Task_DS\PDF_version_1000"
all_pdfs = sorted([f for f in os.listdir(PDF_FOLDER) if f.endswith('.pdf')])

print(f"Processing {len(all_pdfs)} PDFs...")
success = 0
failed = []

for i, filename in enumerate(all_pdfs):
    pdf_path = os.path.join(PDF_FOLDER, filename)
    try:
        result = parse_pdf(pdf_path)
        insert_to_db(conn, result)
        success += 1
        if (i + 1) % 100 == 0:
            print(f"  Progress: {i+1}/{len(all_pdfs)}...")
    except Exception as e:
        failed.append((filename, str(e)))
        print(f"  FAILED: {filename} — {e}")

print(f"\nDone! Success: {success} | Failed: {len(failed)}")
if failed:
    for f, err in failed:
        print(f"  {f}: {err}")

Processing 1000 PDFs...
  Progress: 100/1000...
  Progress: 200/1000...
  Progress: 300/1000...
  Progress: 400/1000...
  Progress: 500/1000...
  Progress: 600/1000...
  Progress: 700/1000...
  Progress: 800/1000...
  Progress: 900/1000...
  Progress: 1000/1000...

Done! Success: 1000 | Failed: 0


In [9]:
#for connecting do database
#DB_PATH = r"C:\Users\asule\Desktop\Task_DS\drilling_reports.db"
#conn = sqlite3.connect(DB_PATH)
#cur = conn.cursor()
#print("Connected!")

Connected!


In [10]:
print("=== DATABASE SUMMARY ===")
tables = ['reports', 'operations', 'equipment_failure',
          'drilling_fluid', 'pore_pressure', 'survey_station',
          'lithology', 'gas_reading', 'stratigraphic']

for table in tables:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    print(f"  {table:<25} = {count:>6} rows")

=== DATABASE SUMMARY ===
  reports                   =   1000 rows
  operations                =  10929 rows
  equipment_failure         =    244 rows
  drilling_fluid            =   1377 rows
  pore_pressure             =    619 rows
  survey_station            =   1035 rows
  lithology                 =    303 rows
  gas_reading               =    316 rows
  stratigraphic             =    168 rows
